# Epi Info AI TABLES validation lab — V0.11

Independently validate categorical TABLES derivation with Python/SciPy and the stratified 2 × 2 adjusted output produced by the Rust/WASM kernel. Passing is engineering evidence, not legacy parity approval.

In [ ]:
# Independent general R x C fixed-margin enumeration; no TypeScript candidate code is called.
import csv, io, math
from pyodide.http import pyfetch
rxc_data_response = await pyfetch('../../examples/foodborne-outbreak-investigation.csv')
rxc_data_response.raise_for_status(); rxc_data_bytes = await rxc_data_response.bytes()
records = list(csv.DictReader(io.StringIO(rxc_data_bytes.decode('utf-8-sig'))))
def bounded_allocations(total, limits, prefix=()):
    if len(limits) == 1:
        if 0 <= total <= limits[0]: yield prefix + (total,)
        return
    remaining_capacity = sum(limits[1:])
    for value in range(max(0, total - remaining_capacity), min(limits[0], total) + 1):
        yield from bounded_allocations(total - value, limits[1:], prefix + (value,))
def fixed_margin_tables(row_margins, column_margins, rows=()):
    if len(row_margins) == 1:
        if sum(column_margins) == row_margins[0]: yield rows + (tuple(column_margins),)
        return
    for row in bounded_allocations(row_margins[0], column_margins):
        yield from fixed_margin_tables(row_margins[1:], tuple(limit - value for limit, value in zip(column_margins, row)), rows + (row,))
def table_probability(table):
    row_margins = [sum(row) for row in table]; column_margins = [sum(column) for column in zip(*table)]; total = sum(row_margins)
    numerator = math.prod(math.factorial(value) for value in row_margins + column_margins)
    denominator = math.factorial(total) * math.prod(math.factorial(value) for row in table for value in row)
    return numerator / denominator
def probability_ordered_exact(table, tolerance=3.45254e-7):
    row_margins = tuple(map(sum, table)); column_margins = tuple(map(sum, zip(*table))); observed = table_probability(table)
    candidates = list(fixed_margin_tables(row_margins, column_margins))
    return sum(table_probability(candidate) for candidate in candidates if table_probability(candidate) <= observed * (1 + tolerance)), len(candidates)
status_values = ['Confirmed', 'Not a case', 'Probable', 'Suspected']; sex_values = ['Female', 'Male']
foodborne_4x2 = [[sum(record['Case Status'] == status and record['Sex'] == sex for record in records) for sex in sex_values] for status in status_values]
assert foodborne_4x2 == [[10, 12], [26, 26], [8, 8], [4, 2]]
foodborne_p, foodborne_tables = probability_ordered_exact(foodborne_4x2)
assert foodborne_tables == 2737 and math.isclose(foodborne_p, 0.8747236693202226, abs_tol=1e-14, rel_tol=0)
three_p, three_tables = probability_ordered_exact([[2, 0, 0], [0, 2, 0], [0, 0, 2]])
assert three_tables == 21 and math.isclose(three_p, 1 / 15, abs_tol=1e-14, rel_tol=0)
print('PASS: independent general R x C enumeration matches foodborne 4 x 2 and synthetic 3 x 3 TABLES anchors')


In [ ]:
import csv, hashlib, io, math
from collections import Counter
from pyodide.http import pyfetch
fixtures = []
for fixture_name in ('foodborne-tables-stratified-v0.3.json', 'foodborne-tables-unstratified-v0.3.json'):
    fixture_response = await pyfetch('../../validation-fixtures/' + fixture_name)
    fixture_response.raise_for_status(); fixtures.append(await fixture_response.json())
data_response = await pyfetch('../../examples/foodborne-outbreak-investigation.csv')
data_response.raise_for_status(); data_bytes = await data_response.bytes()
assert all(hashlib.sha256(data_bytes).hexdigest() == fixture['dataset']['sha256'] for fixture in fixtures)
records = list(csv.DictReader(io.StringIO(data_bytes.decode('utf-8-sig'))))
assert all(len(records) == fixture['dataset']['rows'] == 96 for fixture in fixtures)


In [ ]:
for fixture in fixtures:
    request = fixture['request']
    selected_headers = [request['exposureHeader'], request['outcomeHeader']] + ([request['strataHeader']] if 'strataHeader' in request else [])
    included = [r for r in records if all(r[h].strip() for h in selected_headers)]
    assert len(included) == fixture['includedRecords']
    assert len(records) - len(included) == fixture['excludedMissing']
    for expected_stratum in fixture['strata']:
        members = [r for r in included if 'strataHeader' not in request or r[request['strataHeader']] == expected_stratum['value']]
        counts = Counter((r[request['exposureHeader']], r[request['outcomeHeader']]) for r in members)
        matrix = [[counts[(exposure, outcome)] for outcome in fixture['outcomeValues']] for exposure in fixture['exposureValues']]
        assert matrix == [row['counts'] for row in expected_stratum['rows']]
print('PASS: canonical foodborne records reproduce the unstratified and stratified 2 × 4 count matrices')


In [ ]:
import numpy as np
from scipy.stats import chi2_contingency
for fixture in fixtures:
    for expected_stratum in fixture['strata']:
        observed = np.array([row['counts'] for row in expected_stratum['rows']], dtype=float)
        chi_square, p_value, df, expected = chi2_contingency(observed, correction=False)
        candidate = expected_stratum['pearson']
        assert math.isclose(chi_square, candidate['chiSquare'], abs_tol=1e-12, rel_tol=0)
        assert df == candidate['degreesOfFreedom']
        assert math.isclose(p_value, candidate['pValue'], abs_tol=1e-12, rel_tol=0)
        assert np.allclose(expected, np.array([row['expectedCounts'] for row in expected_stratum['rows']]), atol=1e-12, rtol=0)
        row_percent = observed / observed.sum(axis=1, keepdims=True) * 100
        column_percent = observed / observed.sum(axis=0, keepdims=True) * 100
        assert np.allclose(row_percent, np.array([row['rowPercents'] for row in expected_stratum['rows']]), atol=1e-12, rtol=0)
        assert np.allclose(column_percent, np.array([row['columnPercents'] for row in expected_stratum['rows']]), atol=1e-12, rtol=0)
        assert candidate['cellsExpectedBelowFive'] == int((expected < 5).sum())
        assert candidate['cellsExpectedBelowOne'] == int((expected < 1).sum())
print('PASS: independent SciPy M×N chi-square, expected counts, percentages, and sparse-cell diagnostics match V0.6')


In [ ]:
# Independent Fisher-Freeman-Halton enumeration; this does not call the TypeScript candidate.
import itertools
fisher_response = await pyfetch('../../validation-fixtures/foodborne-tables-fisher-v0.5.json')
fisher_response.raise_for_status(); fisher_fixture = await fisher_response.json()
observed = fisher_fixture['expected']['counts']; margins = [sum(column) for column in zip(*observed)]
row_total = sum(observed[0]); denominator = math.comb(sum(margins), row_total)
def probability(top):
    return math.prod(math.comb(margin, value) for margin, value in zip(margins, top)) / denominator
observed_probability = probability(observed[0]); probabilities = []
for prefix in itertools.product(*(range(margin + 1) for margin in margins[:-1])):
    final = row_total - sum(prefix)
    if 0 <= final <= margins[-1]: probabilities.append(probability((*prefix, final)))
tolerance = fisher_fixture['request']['tolerance']
two_tailed = sum(value for value in probabilities if value <= observed_probability * (1 + tolerance))
assert len(probabilities) == fisher_fixture['expected']['tablesEnumerated'] == 2737
assert math.isclose(two_tailed, fisher_fixture['expected']['twoTailedPValue'], abs_tol=1e-25, rel_tol=0)
print('PASS: independent Python Fisher-Freeman-Halton enumeration matches TABLES V0.6')


In [ ]:
missing_response = await pyfetch('../../validation-fixtures/foodborne-tables-missing-v0.6.json')
missing_response.raise_for_status(); missing_fixture = await missing_response.json()
blank = [record for record in records if not record['Vomiting'].strip() or not record['Sex'].strip()]
assert len(blank) == missing_fixture['expected']['includedMissing'] == 2
missing_counts = Counter(record['Sex'] for record in blank)
assert [missing_counts['Female'], missing_counts['Male']] == missing_fixture['expected']['missingExposureCounts'] == [0, 2]
assert missing_fixture['request']['source'].splitlines() == ['SET (.)="Not recorded"', 'SET MISSING=ON', 'TABLES vomiting Sex', 'SET MISSING=OFF', 'SET (.)="Missing"']
print('PASS: foodborne blanks independently confirm the TABLES V0.8 SET MISSING=ON/OFF fixture')


In [ ]:
# Independently derive the foodborne 2 × 2 strata and core Mantel-Haenszel formulas.
adjusted_response = await pyfetch('../../validation-fixtures/foodborne-tables-adjusted-v0.8.json')
adjusted_response.raise_for_status(); adjusted_fixture = await adjusted_response.json()
request = adjusted_fixture['request']; orientation = request['orientation']
derived_strata = []
for expected_stratum in adjusted_fixture['strata']:
    members = [record for record in records if record[request['strataHeaders'][0]] == expected_stratum['value']]
    cells = [[sum(record[request['exposureHeader']] == exposure and record[request['outcomeHeader']] == outcome for record in members)
              for outcome in (orientation['case'], orientation['nonCase'])]
             for exposure in (orientation['exposed'], orientation['unexposed'])]
    assert cells == expected_stratum['cells']; derived_strata.append((*cells[0], *cells[1]))
sum_ad_over_n = sum(a*d/(a+b+c+d) for a,b,c,d in derived_strata)
sum_bc_over_n = sum(b*c/(a+b+c+d) for a,b,c,d in derived_strata)
mh_or = sum_ad_over_n / sum_bc_over_n
mh_rr = (sum(a*(c+d)/(a+b+c+d) for a,b,c,d in derived_strata) /
         sum(c*(a+b)/(a+b+c+d) for a,b,c,d in derived_strata))
observed_minus_expected = sum(a - (a+b)*(a+c)/(a+b+c+d) for a,b,c,d in derived_strata)
variance = sum((a+b)*(c+d)*(a+c)*(b+d)/((a+b+c+d)**2*(a+b+c+d-1)) for a,b,c,d in derived_strata)
mh_uncorrected = observed_minus_expected**2 / variance
mh_corrected = max(0, abs(observed_minus_expected)-0.5)**2 / variance
expected = adjusted_fixture['adjusted']; tolerance = adjusted_fixture['tolerance']
assert math.isclose(mh_or, expected['adjustedOddsRatio'], abs_tol=tolerance, rel_tol=0)
assert math.isclose(mh_rr, expected['adjustedRiskRatio'], abs_tol=tolerance, rel_tol=0)
assert math.isclose(mh_uncorrected, expected['mantelHaenszelUncorrected'], abs_tol=tolerance, rel_tol=0)
assert math.isclose(mh_corrected, expected['mantelHaenszelCorrected'], abs_tol=tolerance, rel_tol=0)
from scipy.stats import chi2
assert math.isclose(chi2.sf(mh_uncorrected, 1), expected['mantelHaenszelUncorrectedP'], abs_tol=tolerance, rel_tol=0)
print('PASS: independent foodborne derivation and Mantel-Haenszel OR, RR, chi-square, correction, and p-value match TABLES V0.8')


In [ ]:
# Independently reproduce legacy TABLES WEIGHTVAR frequency-weight accumulation.
weighted_response = await pyfetch('../../validation-fixtures/foodborne-tables-weighted-v0.9.json')
weighted_response.raise_for_status(); weighted_fixture = await weighted_response.json()
weighted_exposures = ['No', 'Yes']; weighted_outcomes = ['Confirmed', 'Not a case', 'Probable', 'Suspected']
weighted_cells = [[sum(float(record['Age']) for record in records if record['Potato Salad'] == exposure and record['Case Status'] == outcome)
                   for outcome in weighted_outcomes] for exposure in weighted_exposures]
assert weighted_cells == [row['counts'] for row in weighted_fixture['rows']]
assert sum(map(sum, weighted_cells)) == weighted_fixture['weightedTotal'] == 3917
print('PASS: independent Python WEIGHTVAR frequency weights match TABLES V0.9')


In [ ]:
# Independent PSUVAR Taylor variance reproduction; no TypeScript candidate code is called.
complex_response = await pyfetch('../../validation-fixtures/foodborne-tables-psuvar-v0.2.json')
complex_response.raise_for_status(); complex_fixture = await complex_response.json(); expected = complex_fixture['expected']
working = [{'exposure': r['Potato Salad'], 'outcome': r['Hamburger'], 'stratum': r['Sex'], 'psu': r['Household Neighborhood'], 'weight': float(r['Age'])} for r in records]
def design_variance(items, influence):
    total = 0.0
    for stratum in sorted({r['stratum'] for r in items}):
        members = [r for r in items if r['stratum'] == stratum]; psus = sorted({r['psu'] for r in members})
        if len(psus) <= 1: continue
        q = [sum(influence(r) for r in members if r['psu'] == psu) for psu in psus]
        total += (len(psus) * sum(value * value for value in q) - sum(q) ** 2) / (len(psus) - 1)
    return total
pairs = {(r['stratum'], r['psu']) for r in working}; strata = {r['stratum'] for r in working}; df = len(pairs) - len(strata)
assert len(working) == expected['includedRecords'] == 96 and len(pairs) == expected['primarySamplingUnits'] == 57 and df == expected['degreesOfFreedom'] == 55
assert sum(r['weight'] for r in working) == expected['weightedTotal'] == 3917
from scipy.stats import t
assert math.isclose(t.ppf(0.975, df), expected['confidenceMultiplier'], abs_tol=2e-7, rel_tol=0)
for expected_row in expected['rows']:
    exposure = expected_row['exposureValue']; domain = [r for r in working if r['exposure'] == exposure]; total_weight = sum(r['weight'] for r in domain)
    for expected_cell in expected_row['cells']:
        outcome = expected_cell['outcomeValue']; weighted = sum(r['weight'] for r in domain if r['outcome'] == outcome); p = weighted / total_weight
        variance = design_variance(working, lambda r, exposure=exposure, outcome=outcome, p=p, total_weight=total_weight: r['weight'] * ((1 if r['outcome'] == outcome else 0) - p) / total_weight if r['exposure'] == exposure else 0)
        se = 100 * math.sqrt(variance)
        assert math.isclose(weighted, expected_cell['weightedCount'], abs_tol=1e-12, rel_tol=0)
        assert math.isclose(100*p, expected_cell['rowPercent'], abs_tol=1e-10, rel_tol=0)
        assert math.isclose(se, expected_cell['standardError'], abs_tol=1e-10, rel_tol=0)
        assert math.isclose(100*p - expected['confidenceMultiplier']*se, expected_cell['lowerConfidenceLimit'], abs_tol=1e-10, rel_tol=0)
risk = expected['risk']; a, b = expected['rows'][0]['cells'][0]['weightedCount'], expected['rows'][0]['cells'][1]['weightedCount']; c, d = expected['rows'][1]['cells'][0]['weightedCount'], expected['rows'][1]['cells'][1]['weightedCount']
odds_ratio = a*d/(b*c); risk_ratio = (a/(a+b))/(c/(c+d)); risk_difference = a/(a+b)-c/(c+d)
assert math.isclose(odds_ratio, risk['oddsRatio'], abs_tol=1e-12, rel_tol=0) and math.isclose(risk_ratio, risk['riskRatio'], abs_tol=1e-12, rel_tol=0)
assert math.isclose(100*risk_difference, risk['riskDifferencePercent'], abs_tol=1e-12, rel_tol=0)
out_response = await pyfetch('../../validation-fixtures/foodborne-tables-psuvar-outtable-v0.2.json')
out_response.raise_for_status(); out_fixture = await out_response.json()
assert out_fixture['fields'] == ['potato_salad', 'hamburger', 'COUNT', 'RowPct', 'ColPct', 'StdErr', 'LCL', 'UCL', 'DesignEff']
assert len(out_fixture['records']) == 4
for output_record in out_fixture['records']:
    expected_row = next(row for row in expected['rows'] if row['exposureValue'] == output_record['potato_salad'])
    expected_cell = next(cell for cell in expected_row['cells'] if cell['outcomeValue'] == output_record['hamburger'])
    assert output_record['COUNT'] == expected_cell['count']
    for output_name, cell_name in [('RowPct','rowPercent'), ('ColPct','columnPercent'), ('StdErr','standardError'), ('LCL','lowerConfidenceLimit'), ('UCL','upperConfidenceLimit'), ('DesignEff','designEffect')]:
        assert math.isclose(output_record[output_name], expected_cell[cell_name], abs_tol=1e-10, rel_tol=0)
print('PASS: independent Python PSUVAR Taylor variance, design df/t limits, survey OR/RR/RD, and PSUVAR OUTTABLE match Complex Sample Tables V0.2')


In [ ]:
# Independent CSF Taylor variance + OUTTABLE reproduction; no TypeScript candidate code is called.
csf_response = await pyfetch('../../validation-fixtures/foodborne-frequency-psuvar-v0.1.json')
csf_response.raise_for_status(); csf_fixture = await csf_response.json(); csf_expected = csf_fixture['expected']
csf_working = [{'value': r['Case Status'], 'stratum': r['Sex'], 'psu': r['Household Neighborhood'], 'weight': float(r['Age'])} for r in records]
csf_total = sum(r['weight'] for r in csf_working); csf_values = sorted({r['value'] for r in csf_working})
first_p = sum(r['weight'] for r in csf_working if r['value'] == csf_values[0]) / csf_total
first_variance = design_variance(csf_working, lambda r: r['weight'] * ((1 if r['value'] == csf_values[0] else 0) - first_p) / csf_total)
legacy_deff = first_variance / (first_p * (1-first_p) / (len(csf_working)-1))
for expected_row in csf_expected['rows']:
    value = expected_row['value']; weighted = sum(r['weight'] for r in csf_working if r['value'] == value); p = weighted / csf_total
    variance = design_variance(csf_working, lambda r, value=value, p=p: r['weight'] * ((1 if r['value'] == value else 0) - p) / csf_total)
    se = 100 * math.sqrt(variance); multiplier = csf_expected['confidenceMultiplier']
    assert math.isclose(weighted, expected_row['weightedCount'], abs_tol=1e-12, rel_tol=0)
    assert math.isclose(100*p, expected_row['percent'], abs_tol=1e-10, rel_tol=0)
    assert math.isclose(se, expected_row['standardError'], abs_tol=1e-10, rel_tol=0)
    assert math.isclose(100*p-multiplier*se, expected_row['lowerConfidenceLimit'], abs_tol=1e-10, rel_tol=0)
    assert math.isclose(legacy_deff, expected_row['designEffect'], abs_tol=1e-10, rel_tol=0)
assert csf_fixture['outTable']['fields'] == ['case_status','VARNAME','COUNT','RowPct','ColPct','StdErr','LCL','UCL','DesignEff']
print('PASS: independent Python CSF Taylor variance + OUTTABLE match Complex Sample Frequencies V0.1')


In [ ]:
# Independent CSM Taylor variance reproduction; no TypeScript candidate code is called.
csm_response = await pyfetch('../../validation-fixtures/foodborne-means-psuvar-v0.1.json')
csm_response.raise_for_status(); csm_fixture = await csm_response.json(); csm_expected = csm_fixture['expected']
csm_working = [{'value': float(r['Age']), 'domain': r['Sex'], 'stratum': r['Case Status'], 'psu': r['Household Neighborhood'], 'weight': 1.0} for r in records]
csm_pairs = {(r['stratum'], r['psu']) for r in csm_working}; csm_strata = {r['stratum'] for r in csm_working}; csm_df = len(csm_pairs) - len(csm_strata)
assert len(csm_working) == csm_expected['includedRecords'] == 96 and len(csm_pairs) == csm_expected['primarySamplingUnits'] == 63
assert len(csm_strata) == csm_expected['designStrata'] == 4 and csm_df == csm_expected['degreesOfFreedom'] == 59
assert math.isclose(t.ppf(0.975, csm_df), csm_expected['confidenceMultiplier'], abs_tol=2e-7, rel_tol=0)
csm_estimates = {}
for expected_row in csm_expected['rows'][:2]:
    label = expected_row['label']; members = [r for r in csm_working if r['domain'] == label]; total_weight = sum(r['weight'] for r in members)
    mean = sum(r['value'] * r['weight'] for r in members) / total_weight
    variance = design_variance(csm_working, lambda r, label=label, mean=mean, total_weight=total_weight: (r['value']-mean)*r['weight']/total_weight if r['domain'] == label else 0)
    se = math.sqrt(variance); multiplier = csm_expected['confidenceMultiplier']; csm_estimates[label] = (mean, total_weight)
    assert len(members) == expected_row['count'] and math.isclose(mean, expected_row['mean'], abs_tol=1e-12, rel_tol=0)
    assert math.isclose(se, expected_row['standardError'], abs_tol=1e-10, rel_tol=0)
    assert math.isclose(mean-multiplier*se, expected_row['lowerConfidenceLimit'], abs_tol=1e-10, rel_tol=0)
    assert math.isclose(mean+multiplier*se, expected_row['upperConfidenceLimit'], abs_tol=1e-10, rel_tol=0)
    assert min(r['value'] for r in members) == expected_row['minimum'] and max(r['value'] for r in members) == expected_row['maximum']
left, right = csm_expected['rows'][:2]; difference = csm_expected['rows'][2]; left_mean, left_weight = csm_estimates[left['label']]; right_mean, right_weight = csm_estimates[right['label']]
difference_mean = left_mean-right_mean
difference_variance = design_variance(csm_working, lambda r: (r['value']-left_mean)*r['weight']/left_weight if r['domain'] == left['label'] else -(r['value']-right_mean)*r['weight']/right_weight if r['domain'] == right['label'] else 0)
difference_se = math.sqrt(difference_variance); multiplier = csm_expected['confidenceMultiplier']
assert math.isclose(difference_mean, difference['mean'], abs_tol=1e-12, rel_tol=0) and math.isclose(difference_se, difference['standardError'], abs_tol=1e-10, rel_tol=0)
assert math.isclose(difference_mean-multiplier*difference_se, difference['lowerConfidenceLimit'], abs_tol=1e-10, rel_tol=0)
assert math.isclose(difference_mean+multiplier*difference_se, difference['upperConfidenceLimit'], abs_tol=1e-10, rel_tol=0)
csm_out_response = await pyfetch('../../validation-fixtures/foodborne-means-psuvar-outtable-v0.1.json')
csm_out_response.raise_for_status(); csm_out = await csm_out_response.json()
assert csm_out['status'] == 'new-branch-browser-adaptation'
assert csm_out['fields'] == ['sex','VARNAME','COUNT','MEAN','StdErr','LCL','UCL','MIN','MAX'] and len(csm_out['records']) == 3
for output_record, expected_row in zip(csm_out['records'], csm_expected['rows']):
    assert output_record['sex'] == expected_row['label'] and output_record['VARNAME'] == 'age' and output_record['COUNT'] == expected_row['count']
    for output_name, row_name in [('MEAN','mean'),('StdErr','standardError'),('LCL','lowerConfidenceLimit'),('UCL','upperConfidenceLimit'),('MIN','minimum'),('MAX','maximum')]:
        if expected_row[row_name] is None: assert output_record[output_name] is None
        else: assert math.isclose(output_record[output_name], expected_row[row_name], abs_tol=1e-10, rel_tol=0)
print('PASS: independent Python CSM domain means, Taylor variance, design df/t limits, two-domain difference, and adapted OUTTABLE match Complex Sample Means V0.1')
